# **IMAGE SEARCH SIMILARITY USING FAISS :**

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/caltech-101/caltech-101/scorpion/image_0073.jpg
/kaggle/input/caltech-101/caltech-101/scorpion/image_0040.jpg
/kaggle/input/caltech-101/caltech-101/scorpion/image_0030.jpg
/kaggle/input/caltech-101/caltech-101/scorpion/image_0062.jpg
/kaggle/input/caltech-101/caltech-101/scorpion/image_0020.jpg
/kaggle/input/caltech-101/caltech-101/scorpion/image_0044.jpg
/kaggle/input/caltech-101/caltech-101/scorpion/image_0023.jpg
/kaggle/input/caltech-101/caltech-101/scorpion/image_0050.jpg
/kaggle/input/caltech-101/caltech-101/scorpion/image_0041.jpg
/kaggle/input/caltech-101/caltech-101/scorpion/image_0047.jpg
/kaggle/input/caltech-101/caltech-101/scorpion/image_0070.jpg
/kaggle/input/caltech-101/caltech-101/scorpion/image_0036.jpg
/kaggle/input/caltech-101/caltech-101/scorpion/image_0017.jpg
/kaggle/input/caltech-101/caltech-101/scorpion/image_0068.jpg
/kaggle/input/caltech-101/caltech-101/scorpion/image_0058.jpg
/kaggle/input/caltech-101/caltech-101/scorpion/image_0057.jpg
/kaggle/

## Import libraries :

In [2]:
# Step 0: Install dependencies
!pip install faiss-cpu transformers torch Pillow requests

In [3]:
!pip install langchain-community

In [4]:
!pip install git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-anavmogt
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-anavmogt
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py) ... done


In [5]:
import os 
import faiss
from langchain_community.vectorstores import FAISS
from transformers import CLIPProcessor,CLIPModel

2025-09-24 10:07:08.992886: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758708429.014767     228 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758708429.021486     228 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [6]:
from PIL import Image
import gradio as gr
import clip
from langchain.embeddings import HuggingFaceEmbeddings 

In [7]:
from typing import List, Tuple

## Load images dataset :

In [8]:
def load_images(directory)->Tuple[[List[Image.Image],List[str]]]:
    images_path=[]
    images_list=[]
    for root,dirs,files in os.walk(directory):
        for file in files :
            if file.lower().endswith((".jpg",".jpeg",".png")):
                path=os.path.join(root,file)
            try:
                img=Image.open(path).convert("RGB")
                images_path.append(path)
                images_list.append(img)
            except Exception as e : 
                print(f"Error loading images {e}")
    print(f"Successfully imported images from directory {directory}")
    return images_list,images_path
                

In [9]:
images_list,images_path=load_images("/kaggle/input/caltech-101/caltech-101")

Successfully imported images from directory /kaggle/input/caltech-101/caltech-101


## CLIP Model to convert images into Embeddings :

In [10]:
import torch 
from langchain.embeddings.base import Embeddings 

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model,preprocess=clip.load("ViT-B/32",device=device)

100%|████████████████████████████████████████| 338M/338M [00:03<00:00, 104MiB/s]


In [15]:
class CLIPEmbeddings(Embeddings):
    def __init__(self, model, preprocess, device):
        self.model = model
        self.preprocess = preprocess
        self.device = device

    def embed_documents(self, docs):  # docs is list of PIL images
        # Only preprocess, no Image.open()
        image_tensors = torch.stack([self.preprocess(img).to(self.device) for img in docs])
        with torch.no_grad():
            embeddings = self.model.encode_image(image_tensors)
        embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
        return embeddings.cpu().numpy().tolist()

    def embed_query(self, doc):  # doc is a single PIL image
        img_tensor = self.preprocess(doc).unsqueeze(0).to(self.device)
        with torch.no_grad():
            embedding = self.model.encode_image(img_tensor)
        embedding = embedding / embedding.norm(dim=-1, keepdim=True)
        return embedding.cpu().numpy().flatten().tolist()


In [18]:
def embed_documents_in_batches(clip_embeddings, images_list, batch_size=16):
    all_embeddings = []
    for i in range(0, len(images_list), batch_size):
        batch = images_list[i:i+batch_size]
        batch_tensors = torch.stack([clip_embeddings.preprocess(img).to(clip_embeddings.device) for img in batch])
        with torch.no_grad():
            embeddings = clip_embeddings.model.encode_image(batch_tensors)
            embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
            all_embeddings.extend(embeddings.cpu().numpy())
    return all_embeddings


In [16]:
clip_embeddings=CLIPEmbeddings(model,preprocess,device)

In [19]:
embeddings = embed_documents_in_batches(clip_embeddings, images_list, batch_size=16)
print(f"Total embeddings shape: {len(embeddings)}, each embedding dimension: {len(embeddings[0])}")


Total embeddings shape: 9145, each embedding dimension: 512


In [20]:
import numpy as np

In [21]:
embeddings = np.array(embeddings).astype("float32")
d = embeddings.shape[1]

## FAISS VectorBase Initialization : 

In [32]:
import faiss

dimension = len(embeddings[0])
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings, dtype="float32"))

print(f"Number of vectors in index: {index.ntotal}")


Number of vectors in index: 9145


## Query FAISS with a new image : 

In [35]:
def retrieve_similar_images(user_image):
    # Resize uploaded image
    img = user_image.resize((224, 224))  # <-- Important for CLIP
    
    # Convert into embeddings
    query_embeddings=np.array([clip_embeddings.embed_query(img)],dtype="float32")
    
    # Distances,Indices : 
    distances,indices=index.search(query_embeddings,k=5)
    
     # Printing :
    print("Distances : ",distances)
    print("Indices : ",indices)
    
    # Get matched images :
    matched_images=[images_list[i] for i in indices[0]]
    return matched_images

## Gradio Interface : 

In [36]:
import gradio as gr

# Gradio Interface
iface = gr.Interface(
    fn=retrieve_similar_images,      # function to call
    inputs=gr.Image(type="pil"),   # input is an image
    outputs=gr.Gallery(),          # output is a gallery of images
    title="Image Similarity Search",
    description="Upload an image and find top similar images from the dataset using CLIP + FAISS."
)

# Launch the interface
iface.launch()


* Running on local URL:  http://127.0.0.1:7862
It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://fad2bf04b56e61a313.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Distances :  [[0.03328251 0.30198312 0.3091907  0.31200975 0.3241858 ]]
Indices :  [[4158 4103 4131 4169 4168]]
Distances :  [[0.33339816 0.33365354 0.3640273  0.3778982  0.39839378]]
Indices :  [[3233 3223 7499 3231 7507]]
